In [0]:
%sql
-- LEAD_DIM — lead/student dimension — ALL leads, 1 row per lead_id
-- SPEC (2026-11-11): LEAD_NAME = the verified real client name from Calendly
-- invitee lines (body_text_quoted — the only unmasked name source; reproduces
-- the reference exactly on traced leads Frost/Weed/Welch/Gordon/Sarumathi).
-- NULL when the client never booked; 3_13 falls back to lead_user_hash_name.
-- Multi-name leads: sorted distinct invitee values joined " / " (reference format).
-- The source identity hash is exposed honestly as lead_user_hash_name
-- (3_13 joins the identity spine and staff hashes on that column).
-- Status, ownership (CSM), commercial attributes (contract, tier, avatar).
-- SMS/EMAIL FALLBACK (Nov 12 2026): when a lead has NO Calendly invitee line, LEAD_NAME
-- falls back to the first name staff greeted on OUTBOUND outreach — first SMS
-- ("Hey <Name>, ...", direction='outbound'), then Email body_preview
-- (direction='outgoing' — Close uses a different direction vocabulary per type),
-- e.g. "Hey Brian, Chris here from Data Engineer Academy" — on those rows lead_id
-- is the client and created_by the staff sender, so the greeted name belongs to the
-- lead. Outbound/outgoing only (the inbound mirror greets staff). Fills 141 + 36
-- coaching-client leads whose LEAD_NAME was NULL (never booked; Nov 12 2026
-- measurement). Remaining NULLs: 361 of 1381 clients — no booking, never greeted
-- by name in outbound outreach. 3_13 falls back to lead_user_hash_name for those.
-- Reminder texts ("Hey Damion, your call with Peter starts in 5m...") audited SAFE:
-- they greet the client (the coach is named mid-text, never captured). Greeted name
-- matches the lead's Calendly-verified first name in 95.4% of 1,484 reminder rows —
-- same rate as human-sent outreach (92.2%) and automated regular texts (94.5%).
-- Inbound/incoming greets the STAFF member (100% of inbound greeted names are staff
-- names); outbound greeted name agrees with the Calendly first token on 93% of
-- 7,171 leads that have both. Residual mismatches are nickname/spelling variants
-- of the same person (Rudy/Rudolph, Greg/"Grag", Vikas/"Vikaa") — never merged
-- into one spelling; Calendly priority picks LEAD_NAME, variants stay in all_names.
-- Absorbs the former gold.lead_info names lookup — one lead dimension, one notebook
-- EMAIL ATTRIBUTION (v4, Nov 12 2026): absorbs 3_12_Gold_lead_emails — the columns
-- primary_customer_email / non_primary_customer_emails / customer_emails are
-- built here from the 2_8 identity spine with whole-person staff separation
-- (gold.staff_emails) and CRM-contact-record priority. Population widened to ALL
-- leads (all statuses; status_label kept so 3_13/3_14 filter to Coaching Client).
-- 3_12's gold.lead_emails is retired — notebook removal and consumer migration
-- are deferred job cleanup.
CREATE OR REPLACE TABLE crm_ingestion.gold.lead_dim AS (
WITH leads_processed_deduped AS (
    SELECT
    *
    FROM (
        SELECT
            id AS lead_id,
            name AS lead_user_hash_name,
            display_name AS lead_display_name,
            status_label,
            date_updated,
            bronze_insert_date,
            ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC, bronze_insert_date DESC) AS rn
        FROM
            crm_ingestion.silver.leads_processed
        ) 
        WHERE
            rn = 1
            -- ALL-LEAD GRAIN (Nov 12 2026): keep every lead's LATEST version, ALL
            -- statuses — 1 row per lead_id across the full CRM population ("include
            -- all lead_id", absorbing 3_12's phone-book role). Ranking-first is
            -- preserved so a lead's row always reflects its LATEST status; Current
            -- Coaching Clients = filter status_label = 'Coaching Client'
            -- downstream (3_13/3_14).
)
,
leads_processed_custom_deduped AS (
    SELECT
        *
    FROM (
        SELECT
            leads_processed_id AS lead_id,
            CSM AS csm_id, -- — customer success manager
            Contracted_Value, -- — sales context
            Tier, -- — account info
            Avatar, -- profile image
            bronze_insert_date,
            ROW_NUMBER() OVER (PARTITION BY leads_processed_id ORDER BY bronze_insert_date DESC) AS rn
        FROM
            crm_ingestion.silver.leads_processed_custom
    )
    WHERE
        rn = 1
)
,
-- REAL CLIENT NAME — Calendly invitee lines quoted in email bodies.
invitee_names AS (
    SELECT lead_id,
           ARRAY_JOIN(SORT_ARRAY(COLLECT_SET(name_val)), ' / ') AS real_name,
           COLLECT_SET(name_val) AS all_names
    FROM (
        SELECT lead_id,
               TRIM(REGEXP_EXTRACT(CAST(body_text_quoted AS STRING),
                   '(?s)Invitee:\\s*\\n+\\s*([^\\n]+)', 1)) AS name_val
        FROM crm_ingestion.silver.lead_activites_processed
        WHERE CAST(body_text_quoted AS STRING) LIKE '%Invitee:%'
    )
    WHERE name_val != '' AND name_val NOT LIKE 'user\\_%'
    GROUP BY lead_id
)
,
-- SMS REAL-NAME FALLBACK — outbound staff texts greet the CLIENT by first name
-- ("Hey Brian, Chris here from Data Engineer Academy"). Outbound only: the inbound
-- mirror ("Hey Mary, ...") greets the STAFF member. Stop-greeting tokens excluded,
-- all-lowercase/all-caps captures INITCAP-ed (collapses case-duplicate variants
-- like REDEEMER/Redeemer, THEODOR/Theodor); first names, used only when no Calendly
-- invitee name exists (COALESCE below).
text_names AS (
    SELECT lead_id,
           ARRAY_JOIN(SORT_ARRAY(COLLECT_SET(name_val)), ' / ') AS text_real_name,
           COLLECT_SET(name_val) AS text_all_names
    FROM (
        SELECT lead_id,
               CASE WHEN name_val = LOWER(name_val) OR name_val = UPPER(name_val) THEN INITCAP(name_val) ELSE name_val END AS name_val
        FROM (
            SELECT lead_id,
                   TRIM(REGEXP_EXTRACT(CAST(text AS STRING),
                       '(?:Hey|Hi|Hello)\\s+([A-Za-z]+)\\s*[,!\\.]', 1)) AS name_val
            FROM crm_ingestion.silver.lead_activites_processed
            WHERE type = 'SMS' AND direction = 'outbound'
              AND (text LIKE 'Hey %' OR text LIKE 'Hi %' OR text LIKE 'Hello %')
        )
        WHERE name_val != '' AND name_val NOT LIKE 'user\\_%'
          AND LOWER(name_val) NOT IN ('there','everyone','everybody','friend','buddy','man','dude','brother','bro','boss','team','guys','folks','yes','no','quick','hi','hello','hey','you','it')
    )
    GROUP BY lead_id
)
,
-- EMAIL REAL-NAME FALLBACK — outgoing staff emails greet the CLIENT in the
-- body_preview ("Hi TJ, You reached out to us a while back..."). direction is
-- 'outgoing' for Email (vs 'outbound' for SMS). Same stop-greeting guards; used
-- only when neither a Calendly invitee name nor an SMS-greeted name exists.
email_names AS (
    SELECT lead_id,
           ARRAY_JOIN(SORT_ARRAY(COLLECT_SET(name_val)), ' / ') AS email_real_name,
           COLLECT_SET(name_val) AS email_all_names
    FROM (
        SELECT lead_id,
               CASE WHEN name_val = LOWER(name_val) OR name_val = UPPER(name_val) THEN INITCAP(name_val) ELSE name_val END AS name_val
        FROM (
            SELECT lead_id,
                   TRIM(REGEXP_EXTRACT(CAST(body_preview AS STRING),
                       '(?:Hey|Hi|Hello)\\s+([A-Za-z]+)\\s*[,!\\.]', 1)) AS name_val
            FROM crm_ingestion.silver.lead_activites_processed
            WHERE type = 'Email' AND direction = 'outgoing'
              AND (body_preview LIKE 'Hey %' OR body_preview LIKE 'Hi %' OR body_preview LIKE 'Hello %')
        )
        WHERE name_val != '' AND name_val NOT LIKE 'user\\_%'
          AND LOWER(name_val) NOT IN ('there','everyone','everybody','friend','buddy','man','dude','brother','bro','boss','team','guys','folks','yes','no','quick','hi','hello','hey','you','it')
    )
    GROUP BY lead_id
)
,
-- ===== EMAIL ATTRIBUTION (absorbed from 3_12_Gold_lead_emails, Nov 12 2026) =====
-- The client email view on the 2_8 identity spine. Whole-master staff rule: any
-- email owned by a staff person (48-char organizer login, @dataengineeracademy.com,
-- close_crm registration, or Calendly hosting => gold.staff_emails.all_emails)
-- makes that person staff and subtracts every email they own, masked included.
-- PRIMARY pick priority: 1) REAL email from the lead's own CRM contact record,
-- 2) any REAL non-staff attributed email, 3) MASKED own-record, 4) any MASKED
-- non-staff email; alphabetical tie-break inside each priority.
staff_emails_flat AS (
    SELECT DISTINCT e AS email
    FROM crm_ingestion.gold.staff_emails se
    LATERAL VIEW explode(se.all_emails) e AS e
),
lead_own_emails AS (
    -- the lead's own CRM contact record emails — authoritative for the primary
    SELECT DISTINCT c.leads_processed_id AS lead_id, LOWER(TRIM(ce.email)) AS email,
           (LOWER(TRIM(ce.email)) NOT LIKE '%@masked.com') AS is_real
    FROM crm_ingestion.silver.leads_processed_contacts c
    JOIN crm_ingestion.silver.leads_processed_contacts_emails ce
      ON ce.leads_processed_contacts_id = c.id
    WHERE ce.email IS NOT NULL AND ce.email != ''
),
person_emails AS (
    SELECT explode(sp.lead_ids) AS lead_id, e.email AS email,
           (e.email NOT LIKE '%@masked.com') AS is_real
    FROM crm_ingestion.silver.user_identity_map sp
    LATERAL VIEW explode(concat(sp.real_emails, sp.masked_emails)) e AS email
    WHERE e.email IS NOT NULL AND e.email != ''
),
pe_marked AS (
    SELECT pe.lead_id, pe.email, pe.is_real,
           (s.email IS NOT NULL) AS is_staff,
           (o.lead_id IS NOT NULL) AS is_own
    FROM person_emails pe
    LEFT JOIN staff_emails_flat s ON s.email = pe.email
    LEFT JOIN (SELECT DISTINCT lead_id, email FROM lead_own_emails) o
           ON o.lead_id = pe.lead_id AND o.email = pe.email
),
lead_email_sets AS (
    SELECT lead_id,
           collect_set(CASE WHEN NOT is_staff THEN email END) AS customer_emails
    FROM pe_marked
    GROUP BY lead_id
),
primary_email_pick AS (
    SELECT lead_id, email FROM (
        SELECT lead_id, email,
               ROW_NUMBER() OVER (
                   PARTITION BY lead_id
                   ORDER BY CASE WHEN is_real AND is_own THEN 1
                             WHEN is_real          THEN 2
                             WHEN is_own           THEN 3
                             ELSE 4 END,
                        email
               ) AS rn
        FROM pe_marked WHERE NOT is_staff
    ) WHERE rn = 1
)
SELECT
    l.lead_id,
    -- SPEC: LEAD_NAME is the verified real name (Calendly invitee line), with the
    -- SMS- then email-greeted first name as fallbacks when the client never booked.
    COALESCE(n.real_name, t.text_real_name, e.email_real_name) AS LEAD_NAME,
    l.lead_user_hash_name,
    l.lead_display_name,
    n.all_names,
    t.text_all_names,
    e.email_all_names,
    pep.email AS primary_customer_email,
    CASE WHEN pep.email IS NOT NULL
         THEN array_except(sort_array(les.customer_emails), ARRAY(pep.email))
         ELSE sort_array(les.customer_emails)
    END AS non_primary_customer_emails,
    sort_array(les.customer_emails) AS customer_emails,
    l.status_label,
    l.date_updated,
    l.bronze_insert_date,
    NULLIF(c.csm_id, '\\N') AS csm_id,
    NULLIF(c.Contracted_Value, '\\N') AS Contracted_Value,
    NULLIF(c.Tier, '\\N') AS Tier,
    NULLIF(c.Avatar, '\\N') AS Avatar
FROM
    leads_processed_deduped l
LEFT JOIN
    lead_email_sets les
ON
    l.lead_id = les.lead_id
LEFT JOIN
    primary_email_pick pep
ON
    l.lead_id = pep.lead_id
LEFT JOIN
    leads_processed_custom_deduped c
ON
    l.lead_id = c.lead_id
LEFT JOIN
    invitee_names n
ON
    l.lead_id = n.lead_id
LEFT JOIN
    text_names t
ON
    l.lead_id = t.lead_id
LEFT JOIN
    email_names e
ON
    l.lead_id = e.lead_id
)
